# Baseline 2 — Shared DINOv2 ViT-B/14 backbone

One frozen DINOv2 backbone feeds two trainable classification heads: crop and disease.
Candidate pair score = `log P(crop) + log P(disease)`.

Development uses only the training CSV and its 323 training-derived combinations.
The optional final evaluation reads official tests only after model selection.
Running all cells with default settings does not read official tests.

Run from the repository root or baselines/ folder. Set DATA_ROOT below if needed.
The matching checkpoint and exploratory results are in the
`baseline2-dinov2-base-frozen-v1` GitHub Release.

## Model and training functions

In [ ]:
"""Baseline 2: shared DINOv2 ViT-B/14 backbone and two classification heads."""
import copy
from pathlib import Path
import sys

REPO_ROOT = Path.cwd() if (Path.cwd() / 'common').is_dir() else Path.cwd().parent
if not (REPO_ROOT / 'common').is_dir():
    raise FileNotFoundError('Run from the repository root or baselines/ folder.')
sys.path.insert(0, str(REPO_ROOT))

import timm
import torch
import torch.nn.functional as F
from torch import nn

from common import config
from common.dataset import get_final_test_loaders, get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

# Point this to a directory containing the DPD-Small files if data/ is absent.
DATA_ROOT = None
if DATA_ROOT is not None:
    config.DATA_DIR = Path(DATA_ROOT)
    config.IMAGE_DIR = config.DATA_DIR / 'DPD_Small_dataset'
    config.TRAIN_CSV = config.DATA_DIR / 'DPD_Small_train.csv'
    config.TEST_SEEN_CSV = config.DATA_DIR / 'DPD_Small_test_seen.csv'
    config.TEST_UNSEEN_CSV = config.DATA_DIR / 'DPD_Small_test_unseen.csv'

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
NAME = "baseline2_multitask_dinov2_base_frozen"
PROTOCOL = "training_only_candidates_baseline2_dinov2_frozen_v1"


class MultiTaskNet(nn.Module):
    """One frozen visual encoder, with separate trainable crop and disease heads."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained,
                                          num_classes=0, img_size=224)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, config.NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, config.NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


@torch.no_grad()
def pair_scores(model, images, pairs):
    model.eval()
    crop_logits, disease_logits = model(images.to(config.DEVICE))
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def train(args):
    config.set_seed()
    config.NUM_WORKERS = 0
    batch_size = 8 if args.debug else args.batch_size
    epochs = 1 if args.debug else 20
    train_loader, val_loader, seen, unseen = get_loaders(batch_size, norm="imagenet", debug=args.debug)
    assert seen is None and unseen is None
    train_loader.pin_memory = val_loader.pin_memory = False
    pairs = get_pairs().to(config.DEVICE)
    model = MultiTaskNet(pretrained=True).to(config.DEVICE)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                  lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_val, best_state = -1.0, None
    for epoch in range(epochs):
        # Frozen visual features remain in inference mode; only the heads learn.
        model.eval()
        model.crop_head.train(); model.disease_head.train()
        for images, crop_labels, disease_labels, _ in train_loader:
            images = images.to(config.DEVICE)
            crop_labels = crop_labels.to(config.DEVICE)
            disease_labels = disease_labels.to(config.DEVICE)
            crop_logits, disease_logits = model(images)
            crop_loss = criterion(crop_logits, crop_labels)
            disease_loss = criterion(disease_logits, disease_labels)
            loss = crop_loss + disease_loss
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        scheduler.step()
        top1 = val_accuracy(lambda x: pair_scores(model, x, pairs), val_loader)
        print(f"epoch {epoch + 1}/{epochs}: crop loss {crop_loss.item():.3f}, "
              f"disease loss {disease_loss.item():.3f}, val top-1 {top1:.2f}%", flush=True)
        if top1 > best_val:
            best_val = top1
            best_state = copy.deepcopy(model.state_dict())
    config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    name = NAME + ("_debug" if args.debug else "")
    path = config.CHECKPOINT_DIR / f"{name}.pt"
    hp = {"backbone": BACKBONE, "input_size": 224, "backbone_frozen": True,
          "shared_backbone": True, "batch_size": batch_size, "epochs": epochs,
          "lr": 1e-4, "weight_decay": 1e-4, "disease_loss_weight": 1.0,
          "label_smoothing": 0.1, "debug": args.debug, "protocol": PROTOCOL}
    torch.save({"model_name": name, "state_dict": best_state, "hp": hp,
                "best_validation_top1": best_val}, path)
    print(f"Saved {path}", flush=True)
    print("Training finished. Official tests require a separate final evaluation command.", flush=True)


def final_evaluation(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    hp = checkpoint["hp"]
    if (hp.get("protocol") != PROTOCOL or hp.get("debug")
            or hp.get("backbone") != BACKBONE or not hp.get("shared_backbone")):
        raise ValueError("Use a full Baseline 2 DINOv2 checkpoint from this script")
    config.set_seed()
    config.NUM_WORKERS = 0
    model = MultiTaskNet(pretrained=False)
    model.load_state_dict(checkpoint["state_dict"])
    model.to(config.DEVICE).eval()
    pairs = get_pairs(final_evaluation=True).to(config.DEVICE)
    seen, unseen = get_final_test_loaders(hp["batch_size"], final_evaluation=True)
    evaluate(lambda x: pair_scores(model, x, pairs), seen, unseen,
             checkpoint["model_name"] + "_final", hp, robust=True)


def check():
    import common.dataset as dataset
    original_read = dataset.read_csv
    reads = []

    def guarded(path, **kwargs):
        assert Path(path).resolve() == config.TRAIN_CSV.resolve(), path
        reads.append(str(path))
        return original_read(path, **kwargs)

    dataset.read_csv = guarded
    try:
        train_loader, val_loader, seen, unseen = get_loaders(2, debug=True)
        assert seen is None and unseen is None
        assert get_pairs().shape == (323, 2)
        assert next(iter(train_loader))[0].shape[1:] == (3, 224, 224)
        val_accuracy(lambda images: torch.zeros(len(images), 323), val_loader)
        assert reads
    finally:
        dataset.read_csv = original_read
    model = MultiTaskNet(pretrained=False)
    assert model.backbone.patch_embed.img_size == (224, 224)
    assert {name for name, p in model.named_parameters() if p.requires_grad} == {
        "crop_head.weight", "crop_head.bias", "disease_head.weight", "disease_head.bias"}
    print("PASS: one DINOv2 backbone, two trainable heads, training CSV only, 323 development candidates.")



## Train`n`nSet RUN_TRAINING to False to load an existing checkpoint without retraining.

In [ ]:
from types import SimpleNamespace

RUN_TRAINING = True
DEBUG = False
BATCH_SIZE = 16
if RUN_TRAINING:
    train(SimpleNamespace(debug=DEBUG, batch_size=BATCH_SIZE))

## Optional final evaluation`n`nSet RUN_FINAL_EVALUATION to True only after model choices are frozen. Prior official test results have been inspected, so report these as exploratory and do not tune with them.

In [ ]:
RUN_FINAL_EVALUATION = False
CHECKPOINT_PATH = config.CHECKPOINT_DIR / f'{NAME}.pt'
if RUN_FINAL_EVALUATION:
    final_evaluation(CHECKPOINT_PATH)